# Evaluate on a Colab A100

This notebook is a batch job. It serves the baseline and candidate with vLLM on this GPU, runs the selected suite through `gate run`, stops both servers, and copies `runs/<run_id>/` to Google Drive. Scoring and the gate decision run locally on the saved run.

Requirements:
- A Colab A100 runtime.
- A Colab secret named `HF_TOKEN`. Gated models such as Llama also need the license accepted and access approved on Hugging Face before a run; cell 6 checks this before any server starts.

Compute units are billed while the runtime is connected. Run all cells in order (Drive sign-in happens early, before the GPU servers start), then disconnect as the last cell says.

In [ ]:
REPO_URL = "https://github.com/abhishek-paul-d/LLM_GATE.git"
BRANCH = "main"
BASELINE = "llama-3.1-8b-instruct-fp8"  # or "ministral-3-8b-instruct-fp8" (ungated)
CANDIDATE = "qwen3-8b-fp8"
SUITE = "suites/starter-v1"
SPLIT = "release"
PROMPT = "prompts/triage-v1.yaml"
PORTS = {"baseline": 8001, "candidate": 8002}
GPU_MEMORY_UTILIZATION = 0.45
# Pinned: vLLM 0.30.0 (Sep 2026) routes online FP8 on an A100 (no native FP8) to a CUTLASS kernel
# that fails (cutlass_scaled_mm_sm80_epilogue); 0.11.x uses the weight-only Marlin fallback.
# 0.11.2 pins torch 2.9.0 with a matching torchaudio and transformers<5. The manifest records it.
VLLM_VERSION = "0.11.2"
DRIVE_DIR = "/content/drive/MyDrive/llm-release-gate/runs"

In [ ]:
!nvidia-smi

In [ ]:
import os

if not os.path.isdir("/content/LLM_GATE"):
    !git clone --branch {BRANCH} {REPO_URL} /content/LLM_GATE
%cd /content/LLM_GATE
!pip install -q -e . vllm=={VLLM_VERSION}
# If vLLM's torch and Colab's preinstalled torchaudio are built for different CUDA versions,
# transformers imports torchaudio and `vllm serve` crashes at startup ("PyTorch and TorchAudio
# were compiled with different CUDA versions"; seen with unpinned vLLM). Text models don't need it.
!pip uninstall -q -y torchaudio

# The editable install adds src/ to sys.path through a .pth file, which Python reads only at
# startup. This kernel started before the install, so add src/ by hand. Subprocesses such as
# `gate run` start fresh and don't need this.
import sys

if "/content/LLM_GATE/src" not in sys.path:
    sys.path.insert(0, "/content/LLM_GATE/src")
import release_gate  # noqa: E402, F401  (fails here, not later, if the install did not work)

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Mount Drive now, before the GPU servers start, so the sign-in prompt never holds up a billed session later.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import torch
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError

from release_gate import registry

baseline_spec = registry.load_model_spec(BASELINE)
candidate_spec = registry.load_model_spec(CANDIDATE)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
specs = [baseline_spec, candidate_spec]
# Refuse before any server starts: the manifest records the revision as given, so a branch
# such as "main" would not identify the weights that were served.
unpinned = [spec.name for spec in specs if not spec.revision_pinned]
if unpinned:
    raise RuntimeError(f"Pin model.revision to a 40-char commit SHA in models/<name>.yaml first: {unpinned}")
# Check Hugging Face access now: a gated repo the token's account is not approved for would
# otherwise only fail inside `vllm serve`. config.json is a few KB and lands in the HF cache.
for spec in specs:
    try:
        hf_hub_download(spec.model.id, "config.json", revision=spec.model.revision)
    except GatedRepoError as error:
        raise RuntimeError(
            f"HF_TOKEN's account has no access to {spec.model.id} yet. Request it at "
            f"https://huggingface.co/{spec.model.id} and wait for approval before reconnecting a GPU."
        ) from error
if not registry.fits_concurrently(specs, gpu_gb):
    raise RuntimeError(
        "These models do not fit concurrently. Sequential mode is not implemented yet; use smaller or quantized specs."
    )
print("Revisions:", {spec.name: spec.model.revision[:12] for spec in specs})
print(f"Combined model weights: {sum(spec.serving.weights_gb for spec in specs):.1f} GB; GPU memory: {gpu_gb:.1f} GB")

In [ ]:
import importlib.metadata
import json
import subprocess
from pathlib import Path

nvidia_smi = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
gpu_name, gpu_memory_total, driver_version = [part.strip() for part in nvidia_smi.split(",", 2)]
environment = {
    "gpu_name": gpu_name,
    "gpu_memory_total": gpu_memory_total,
    "driver_version": driver_version,
    "cuda_version": str(torch.version.cuda or "unknown"),
    "torch_version": str(torch.__version__),
    "vllm_version": importlib.metadata.version("vllm"),
    "execution_mode": "concurrent",
    "gpu_memory_utilization": str(GPU_MEMORY_UTILIZATION),
}
Path("/content/env.json").write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")
environment

In [ ]:
import time

import httpx

Path("/content/logs").mkdir(parents=True, exist_ok=True)
servers = {}
log_files = {}


def wait_healthy(role, port, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    url = f"http://127.0.0.1:{port}/v1/models"
    while time.monotonic() < deadline:
        if servers[role].poll() is not None:
            break
        try:
            if httpx.get(url, timeout=5).status_code == 200:
                print(f"{role} server is healthy")
                return
        except httpx.HTTPError:
            pass
        time.sleep(5)
    log_path = Path(f"/content/logs/{role}.log")
    lines = log_path.read_text(errors="replace").splitlines() if log_path.exists() else []
    # vLLM's API server only reports "Engine core initialization failed. See root cause above";
    # the root cause is the engine process's first error, so print from there, not just the tail.
    first_error = next((i for i, line in enumerate(lines) if "ERROR" in line or "Traceback" in line), None)
    shown = lines[first_error : first_error + 80] if first_error is not None else lines[-80:]
    print(f"{role} server log ({log_path}), from the first error:\n" + "\n".join(shown))
    exit_code = servers[role].poll()
    status = f"exited with code {exit_code}" if exit_code is not None else f"was not healthy after {timeout_s} s"
    raise RuntimeError(f"{role} server {status}; full log: {log_path}")


def start_server(role, spec):
    log_file = open(f"/content/logs/{role}.log", "w")
    log_files[role] = log_file
    command = spec.vllm_serve_args(PORTS[role], GPU_MEMORY_UTILIZATION)
    servers[role] = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, env=os.environ.copy())


def stop_servers():
    for process in servers.values():
        if process.poll() is None:
            process.terminate()
    for process in servers.values():
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    for log_file in log_files.values():
        log_file.close()
    servers.clear()
    log_files.clear()
    print("vLLM servers stopped.")

In [ ]:
# Start the baseline, wait until it is healthy, then the candidate, so the two servers do not race
# for GPU memory. The finally block stops both servers even if a health check or the run fails.
from datetime import UTC, datetime

RUN_ID = f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}-{BASELINE}-vs-{CANDIDATE}"
RUN_DIR = Path("runs") / RUN_ID
try:
    start_server("baseline", baseline_spec)
    wait_healthy("baseline", PORTS["baseline"])
    start_server("candidate", candidate_spec)
    wait_healthy("candidate", PORTS["candidate"])
    subprocess.run(
        [
            "gate",
            "run",
            "--suite",
            SUITE,
            "--split",
            SPLIT,
            "--prompt",
            PROMPT,
            "--baseline",
            BASELINE,
            "--candidate",
            CANDIDATE,
            "--baseline-url",
            f"http://127.0.0.1:{PORTS['baseline']}/v1",
            "--candidate-url",
            f"http://127.0.0.1:{PORTS['candidate']}/v1",
            "--env-file",
            "/content/env.json",
            "--run-id",
            RUN_ID,
        ],
        check=True,
    )
finally:
    stop_servers()
print("Saved run:", RUN_DIR)

In [ ]:
import shutil

destination = Path(DRIVE_DIR) / RUN_ID
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_DIR, destination)
print("Copied run to", destination)

## Disconnect now

Use **Runtime > Disconnect and delete runtime** now to stop compute-unit billing. The run directory is on Google Drive and can be scored and gated locally.